In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
sentences = [
    "Machine learning is interesting",
    "I love learning about AI",
    "I cooked food for dinner"
]

embeddings = model.encode(sentences)

print("Shape:", embeddings.shape)

Shape: (3, 384)


In [3]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)

print(similarity_matrix)

[[1.         0.6096147  0.0735136 ]
 [0.6096147  1.0000002  0.16801842]
 [0.0735136  0.16801842 1.0000001 ]]


In [4]:
documents = [
    "Machine learning uses algorithms to learn patterns from data.",
    "Transformers use attention mechanisms to process sequences.",
    "RAG retrieves relevant documents before generating an answer.",
    "The semiconductor experiment increased temperature to 300 degrees.",
    "The experiment showed increased transistor leakage current."
]

query = "What happened when the semiconductor experiment temperature increased?"

document_embeddings = model.encode(documents)
query_embedding = model.encode([query])

similarities = cosine_similarity(
    query_embedding,
    document_embeddings
)[0]

for i, score in enumerate(similarities):
    print(f"{score:.4f} -> {documents[i]}")

0.0491 -> Machine learning uses algorithms to learn patterns from data.
0.0645 -> Transformers use attention mechanisms to process sequences.
0.0578 -> RAG retrieves relevant documents before generating an answer.
0.8688 -> The semiconductor experiment increased temperature to 300 degrees.
0.5257 -> The experiment showed increased transistor leakage current.


In [5]:
import numpy as np

top_k = 2

top_indices = np.argsort(similarities)[::-1][:top_k]

print("Top", top_k, "documents:\n")

for rank, index in enumerate(top_indices, start=1):
    print(f"Rank {rank}")
    print(f"Score: {similarities[index]:.4f}")
    print(f"Document: {documents[index]}")
    print()

Top 2 documents:

Rank 1
Score: 0.8688
Document: The semiconductor experiment increased temperature to 300 degrees.

Rank 2
Score: 0.5257
Document: The experiment showed increased transistor leakage current.



## RAG Practical

In [6]:
document = """
Experiment 27 was conducted to study transistor behavior at different temperatures.

The initial temperature was 200 degrees Celsius. The transistor showed normal leakage current at this temperature.

The temperature was then increased to 250 degrees Celsius. A small increase in leakage current was observed.

The temperature was finally increased to 300 degrees Celsius. A significant increase in transistor leakage current was observed.

The experiment concluded that transistor leakage current increases significantly at higher temperatures.
"""

In [7]:
chunks = [
    "Experiment 27 was conducted to study transistor behavior at different temperatures.",

    "The initial temperature was 200 degrees Celsius. The transistor showed normal leakage current at this temperature.",

    "The temperature was then increased to 250 degrees Celsius. A small increase in leakage current was observed.",

    "The temperature was finally increased to 300 degrees Celsius. A significant increase in transistor leakage current was observed.",

    "The experiment concluded that transistor leakage current increases significantly at higher temperatures."
]

for i, chunk in enumerate(chunks):
    print(f"Chunk {i}:")
    print(chunk)
    print()

Chunk 0:
Experiment 27 was conducted to study transistor behavior at different temperatures.

Chunk 1:
The initial temperature was 200 degrees Celsius. The transistor showed normal leakage current at this temperature.

Chunk 2:
The temperature was then increased to 250 degrees Celsius. A small increase in leakage current was observed.

Chunk 3:
The temperature was finally increased to 300 degrees Celsius. A significant increase in transistor leakage current was observed.

Chunk 4:
The experiment concluded that transistor leakage current increases significantly at higher temperatures.



In [8]:
chunk_embeddings = model.encode(chunks)

print("Shape:", chunk_embeddings.shape)

Shape: (5, 384)


In [9]:
print(chunk_embeddings[0])

[-4.08351198e-02 -1.38359023e-02 -2.77078655e-02  8.65908191e-02
 -5.35514243e-02 -8.05266127e-02 -4.11706157e-02  4.19809557e-02
 -1.54542984e-04  2.07674168e-02  2.94962320e-02 -6.93512782e-02
 -6.28624950e-03  3.17343292e-05  2.14668009e-02 -1.19545273e-02
  1.30498130e-02 -9.61164758e-03  6.53903000e-03 -7.84501061e-02
 -9.02436115e-03 -5.18685058e-02  3.05465925e-02  8.31431418e-04
 -6.96207806e-02  3.86041962e-02  6.17185496e-02  9.37923230e-03
 -3.31505574e-02 -8.09721928e-03 -5.63400835e-02  1.65809244e-02
 -1.49733818e-03  6.70874193e-02 -4.56866995e-02 -2.75758263e-02
 -4.13810015e-02 -5.29561527e-02  2.52387226e-02  1.34445436e-03
 -5.74853383e-02 -8.18297416e-02  1.13600709e-01  1.22915739e-02
  7.57955313e-02 -4.31512706e-02 -3.18133160e-02 -8.22734088e-02
 -4.44951206e-02 -6.49447888e-02  5.04118325e-05 -1.05326129e-02
  6.16021343e-02  3.89666483e-02 -5.90384342e-02  2.57129297e-02
  2.76688207e-02  3.17611434e-02  2.79446635e-02  5.65834045e-02
  5.48891313e-02 -2.04737

In [10]:
query = "What happened when the temperature reached 300 degrees?"

query_embedding = model.encode([query])

similarities = cosine_similarity(
    query_embedding,
    chunk_embeddings
)[0]

for i, score in enumerate(similarities):
    print(f"Chunk {i}: {score:.4f}")
    print(chunks[i])
    print()

Chunk 0: 0.3677
Experiment 27 was conducted to study transistor behavior at different temperatures.

Chunk 1: 0.5240
The initial temperature was 200 degrees Celsius. The transistor showed normal leakage current at this temperature.

Chunk 2: 0.5721
The temperature was then increased to 250 degrees Celsius. A small increase in leakage current was observed.

Chunk 3: 0.5778
The temperature was finally increased to 300 degrees Celsius. A significant increase in transistor leakage current was observed.

Chunk 4: 0.3737
The experiment concluded that transistor leakage current increases significantly at higher temperatures.



## Top-K Retrieval

In [11]:
top_k = 3

top_indices = np.argsort(similarities)[::-1][:top_k]

for rank, i in enumerate(top_indices, start=1):
    print(f"Rank {rank}")
    print(f"Chunk {i}")
    print(f"Similarity: {similarities[i]:.4f}")
    print(chunks[i])
    print()

Rank 1
Chunk 3
Similarity: 0.5778
The temperature was finally increased to 300 degrees Celsius. A significant increase in transistor leakage current was observed.

Rank 2
Chunk 2
Similarity: 0.5721
The temperature was then increased to 250 degrees Celsius. A small increase in leakage current was observed.

Rank 3
Chunk 1
Similarity: 0.5240
The initial temperature was 200 degrees Celsius. The transistor showed normal leakage current at this temperature.



In [15]:
retrieved_chunks = [chunks[i] for i in top_indices]

context = "\n\n".join(retrieved_chunks)

print(context)

The temperature was finally increased to 300 degrees Celsius. A significant increase in transistor leakage current was observed.

The temperature was then increased to 250 degrees Celsius. A small increase in leakage current was observed.

The initial temperature was 200 degrees Celsius. The transistor showed normal leakage current at this temperature.


## Step 1 — Prepare the question + retrieved context

In [12]:
question = "What happened when the temperature reached 300 degrees?"

retrieved_chunks = [chunks[i] for i in top_indices]

context = "\n\n".join(retrieved_chunks)

In [13]:
prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{question}

Answer:
"""

print(prompt)


Answer the question using only the context below.

Context:
The temperature was finally increased to 300 degrees Celsius. A significant increase in transistor leakage current was observed.

The temperature was then increased to 250 degrees Celsius. A small increase in leakage current was observed.

The initial temperature was 200 degrees Celsius. The transistor showed normal leakage current at this temperature.

Question:
What happened when the temperature reached 300 degrees?

Answer:



In [14]:
import os

print("API key configured:", bool(os.getenv("OPENAI_API_KEY")))

API key configured: True


In [15]:
from openai import OpenAI

client = OpenAI()

question = "What happened when the temperature reached 300 degrees?"

retrieved_chunks = [chunks[i] for i in top_indices]

context = "\n\n".join(retrieved_chunks)

prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{question}

Answer:
"""

response = client.responses.create(
    model="gpt-5.6",
    input=prompt
)

print(response.output_text)

A significant increase in transistor leakage current was observed.


##  FAISS Vector Store 

In [20]:
import faiss
import numpy as np

In [21]:
chunk_embeddings = model.encode(chunks)

print("Shape:", chunk_embeddings.shape)

Shape: (5, 384)


In [22]:
embeddings = np.asarray(chunk_embeddings).astype("float32")

print(embeddings.shape)

(5, 384)


In [23]:
# Normalize vectors for cosine similarity
faiss.normalize_L2(embeddings)

# Vector dimension
dimension = embeddings.shape[1]

# Create FAISS index
index = faiss.IndexFlatIP(dimension)

# Add all chunk embeddings
index.add(embeddings)

print("Number of vectors:", index.ntotal)

Number of vectors: 5


##  Create a query embeddings

In [ ]:
question = "What happened when the temperature reached 300 degrees?"

query_embedding = model.encode([question]).astype("float32") 

faiss.normalize_L2(query_embedding)

## Search FAISS for Top-3

In [25]:
scores, indices = index.search(query_embedding, 3)

print("Scores:", scores)
print("Indices:", indices)

Scores: [[0.5778056 0.5720992 0.5239837]]
Indices: [[3 2 1]]


## Map the results back to the original chunks

In [26]:
for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
    print(f"Rank {rank}")
    print(f"Chunk {idx}")
    print(f"Similarity: {score:.4f}")
    print(chunks[idx])
    print()

Rank 1
Chunk 3
Similarity: 0.5778
The temperature was finally increased to 300 degrees Celsius. A significant increase in transistor leakage current was observed.

Rank 2
Chunk 2
Similarity: 0.5721
The temperature was then increased to 250 degrees Celsius. A small increase in leakage current was observed.

Rank 3
Chunk 1
Similarity: 0.5240
The initial temperature was 200 degrees Celsius. The transistor showed normal leakage current at this temperature.



## Chroma

In [16]:
import chromadb

In [17]:
chroma_client = chromadb.Client()

collection = chroma_client.create_collection(
    name="experiment_docs"
)

print("Collection created")

Collection created


In [18]:
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    embeddings=chunk_embeddings.tolist(),
    documents=chunks,
    metadatas=[
        {"document_id": "experiment_27", "chunk_id": i}
        for i in range(len(chunks))
    ]
)

In [19]:
print(collection.count())

5


In [20]:
question = "What happened when the temperature reached 300 degrees?"

query_embedding = model.encode([question])[0].tolist()

In [21]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

In [22]:
print("Documents:")
for doc in results["documents"][0]:
    print(doc)
    print()

print("IDs:")
print(results["ids"][0])

print("Metadata:")
print(results["metadatas"][0])

print("Distances:")
print(results["distances"][0])

Documents:
The temperature was finally increased to 300 degrees Celsius. A significant increase in transistor leakage current was observed.

The temperature was then increased to 250 degrees Celsius. A small increase in leakage current was observed.

The initial temperature was 200 degrees Celsius. The transistor showed normal leakage current at this temperature.

IDs:
['chunk_3', 'chunk_2', 'chunk_1']
Metadata:
[{'document_id': 'experiment_27', 'chunk_id': 3}, {'document_id': 'experiment_27', 'chunk_id': 2}, {'chunk_id': 1, 'document_id': 'experiment_27'}]
Distances:
[0.8443889021873474, 0.8558015823364258, 0.9520326852798462]
